# Model Comparison

The purpose of this notebook is to compare the baseline models trained in the previous step and to assess how well they perform on the financial phrase sentiment classification task.
Because the dataset is imbalanced, accuracy alone does not provide a reliable picture of model quality.
To obtain a more balanced assessment, the models are evaluated using macro‑F1 and weighted‑F1 scores, which reflect performance across all sentiment categories and account for class imbalance.
The goal is to identify the strongest baseline model and to understand the relative strengths and limitations of each classifier.

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve()
while not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.append(str(project_root))

In [23]:
from src.utils.utils import load_metrics
import pandas as pd

pd.set_option('display.max_colwidth', None)

## Understanding the Evaluation Metrics

Machine learning models for text classification are typically evaluated using precision, recall, and F1‑score. These metrics provide a more nuanced view than accuracy, especially when the dataset is imbalanced — which is the case here.


Accuracy measures the proportion of correctly predicted samples:

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

While easy to interpret, accuracy can be misleading in imbalanced datasets.
In this project, the neutral class is much more frequent than the negative class.
A model that predicts “neutral” for everything would still achieve over 60% accuracy.


Precision answers the question: Of all samples predicted as class X, how many were actually class X?

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

High precision means few false positives.


Recall answers the question of all actual samples of class X, how many did the model correctly identify?

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

High recall means few false negatives.


The F1‑score is the harmonic mean of precision and recall:

$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

It balances both types of errors and is especially useful when classes are imbalanced and both false positives and false negatives matter.


Macro‑F1 computes the F1‑score for each class independently and then averages them:

$$
\text{Macro-F1} = \frac{1}{K} \sum_{i=1}^{K} F1_i
$$

Every class counts equally, regardless of how many samples it has. It reveals if a model performs poorly on minority classes.


Weighted‑F1 averages the F1‑scores but weights them by class frequency:

$$
\text{Weighted-F1} = \sum_{i=1}^{K} \left( \frac{\text{support}_i}{\text{total}} \cdot F1_i \right)
$$

This metric reflects overall performance but is dominated by the majority class.


Macro‑F1 is chosen as the main metric because it gives each class equal weight and therefore reflects performance more reliably on our imbalanced sentiment dataset.

## Loading the Model Results

Each baseline model produced a set of evaluation metrics during training.
In this section, these results are collected into a single overview to enable a clear and consistent comparison of model performance across all sentiment classes.

In [24]:
df = load_metrics()
df

,model,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,config
0,logistic_regression,0.867550,0.817436,0.867594,0.813197,0.825124,"{'model': 'Logistic Regression', 'params': {'C': 1.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}, 'tfidf': {'max_features': 3000, 'ngram_range': [1, 2]}}"
1,naive_bayes,0.801325,0.654826,0.778570,0.833301,0.641588,"{'model': 'Naive Bayes', 'params': {'alpha': 1.0, 'class_prior': None, 'fit_prior': True, 'force_alpha': True}, 'tfidf': {'max_features': 3000, 'ngram_range': [1, 2]}}"
2,linear_svm,0.898455,0.851510,0.898199,0.849079,0.857341,"{'model': 'Linear SVM', 'params': {'C': 1.0, 'class_weight': 'balanced', 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 1000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 42, 'tol': 0.0001, 'verbose': 0}, 'tfidf': {'max_features': 3000, 'ngram_range': [1, 2]}}"


## Overview of Model Performance
With all evaluation results collected in a single DataFrame, the baseline models can now be viewed side by side.
This initial overview highlights the differences in performance across the models and provides a first indication of which approaches handle the sentiment classes most effectively.
Sorting the results by the macro‑F1 score places the focus on the metric that best reflects balanced performance on the imbalanced dataset.

In [25]:
df[["model", "accuracy", "macro_f1", "weighted_f1"]].sort_values(
    by="macro_f1", ascending=False
)

,model,accuracy,macro_f1,weighted_f1
2,linear_svm,0.898455,0.851510,0.898199
0,logistic_regression,0.867550,0.817436,0.867594
1,naive_bayes,0.801325,0.654826,0.778570


The comparison shows clear performance differences between the three baseline models.
The linear SVM achieves the strongest results across all metrics. Its macro‑F1 score of 0.85 indicates that it handles all sentiment classes consistently well, including the minority classes. The high weighted‑F1 and accuracy values confirm that this performance is stable across the dataset.

The logistic regression model performs slightly weaker but remains competitive. With a macro‑F1 of 0.82, it captures the class structure reasonably well, although its performance on the minority classes is less balanced than that of the SVM.

The naive Bayes model shows the largest performance gap. Its macro‑F1 score of 0.65 suggests difficulties in distinguishing between the sentiment classes, particularly the less frequent ones. While its accuracy remains above 0.80, this is mainly driven by the majority class and does not reflect balanced performance.

Overall, the results highlight the linear SVM as the most effective baseline model, offering the best combination of balanced class performance and overall predictive strength.